# nb3b — Baseline Identity & Zero-shot BARTpho-syllable (đối chứng cho nb3)

Notebook phụ trợ của `nb3-baseline-train-eval` (`DESIGN.md §7`), chạy **chỉ eval, không train**, đo 2 baseline đối chứng trên **cả VSEC-val (927 câu)** và **test_aligned (~6.000 câu)**:
1. **Identity baseline**: `pred = src` (không sửa gì) — sàn (floor) lý thuyết của mọi mô hình.
2. **Zero-shot baseline**: `vinai/bartpho-syllable` gốc fp16, greedy decode — **không LoRA, không train**.

Đầy đủ 3 nhóm chỉ số bắt buộc (`PROJECT.md §6`): Detection P/R/F1, Correction Accuracy @TP, Over-correction Rate + Clean Retention, kèm phân tích stratified non-word/real-word. Kết quả trả lời 2 câu hỏi:
- "LoRA fine-tune đóng góp bao nhiêu?" → so Zero-shot với Run 1/Run 2 của nb3.
- "Over-correction ~1,8% là do fine-tune hay base model có sẵn?" → so Over-correction của Zero-shot với Run 1/Run 2.

**Input** (attach vào Kaggle):
- Bắt buộc: `vsec_val.jsonl` (từ nb0), `test_aligned.jsonl` (từ nb1).
- Tùy chọn: `eval_report.json` (output của nb3 — để in bảng so sánh 4-way `Identity | Zero-shot | Run 1 | Run 2`), `syllable_table.json` (từ nb2 — cho stratified). Thiếu input tùy chọn thì tự bỏ qua tính năng tương ứng, không crash.

**Output** (ghi `/kaggle/working`):
- `predictions_val_zeroshot.jsonl`, `predictions_test_zeroshot.jsonl`
- `zeroshot_eval_report.json` (file riêng, KHÔNG ghi đè `eval_report.json` của nb3)

**Quy tắc chống rò rỉ dữ liệu (Anti-leakage — DESIGN.md §9)**: notebook chỉ nạp val/test để đánh giá; không nạp `vsec_train.jsonl`, không fit bất kỳ thống kê hay tham số nào → anti-leakage hiển nhiên đúng.

**Vận hành trên Kaggle** (~25–35 phút tổng, GPU T4):
1. Tạo Kaggle notebook mới, bật GPU T4.
2. Attach Input: dataset chứa `vsec_val.jsonl` + `test_aligned.jsonl` như nb3 đã dùng; **thêm output của nb3 làm Input** để lấy `eval_report.json` cho bảng 4-way (nếu bộ Input cũ đã có `syllable_table.json` thì cứ dùng).
3. Upload notebook, Run All (val ~3 phút, test ~15–25 phút, còn lại load model/data).
4. Tải về `/kaggle/working`: 2 file predictions + `zeroshot_eval_report.json`.


In [1]:
import os
import json
import datetime
import unicodedata
from pathlib import Path

# Cài đặt thư viện cần thiết nếu chạy trên Kaggle (notebook chỉ eval, không train — không cần adapter/trainer)
try:
    import sentencepiece
except ImportError:
    print('Cài đặt sentencepiece...')
    os.system('pip install -q sentencepiece')

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, set_seed

SEED = 42
set_seed(SEED)

MODEL_NAME = 'vinai/bartpho-syllable'
EVAL_BEAM_SIZE = 1  # Greedy decode — determinism: cùng weights ⇒ cùng kết quả
MAX_SOURCE_LEN = 256
MAX_TARGET_LEN = 256
BATCH_GEN = 16

FP16 = torch.cuda.is_available()

OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path('./out')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_STAMP = datetime.datetime.now().isoformat(timespec='seconds')

REQUIRED_FILES = ['vsec_val.jsonl', 'test_aligned.jsonl']
OPTIONAL_FILES = ['eval_report.json', 'syllable_table.json']

def find_required_inputs():
    candidates = []
    kaggle = Path('/kaggle/input')
    if kaggle.is_dir():
        candidates += sorted(kaggle.rglob('*.jsonl')) + sorted(kaggle.rglob('*.json'))
    local = Path('./out')
    if local.is_dir():
        candidates += sorted(local.rglob('*.jsonl')) + sorted(local.rglob('*.json'))

    found = {}
    for p in candidates:
        name = p.name
        if name in REQUIRED_FILES + OPTIONAL_FILES:
            if name not in found:
                found[name] = p

    missing = [req for req in REQUIRED_FILES if req not in found]
    if missing:
        raise FileNotFoundError(f'Thiếu các tệp đầu vào bắt buộc: {missing}. Hãy kiểm tra Input Datasets trên Kaggle hoặc thư mục ./out!')
    return found

INPUT_FILES = find_required_inputs()
print('=== TỰ DÒ INPUT HOÀN TẤT ===')
for k, v in INPUT_FILES.items():
    tag = '(bắt buộc)' if k in REQUIRED_FILES else '(tùy chọn)'
    print(f'  {k:20s}: {v} {tag}')
missing_optional = [o for o in OPTIONAL_FILES if o not in INPUT_FILES]
if missing_optional:
    print(f'  Cảnh báo: thiếu input tùy chọn {missing_optional} — sẽ tự bỏ qua tính năng tương ứng.')
print(f'Device: {"CUDA " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"} | FP16={FP16}')
if not FP16:
    print('Cảnh báo: không có CUDA → nạp model float32 (chạy sẽ rất chậm).')


=== TỰ DÒ INPUT HOÀN TẤT ===
  vsec_val.jsonl      : /kaggle/input/notebooks/cquangnguynl/nb0-data-pre/vsec_val.jsonl (bắt buộc)
  test_aligned.jsonl  : /kaggle/input/notebooks/cquangnguynl/nb1-align-annotate/test_aligned.jsonl (bắt buộc)
  syllable_table.json : /kaggle/input/notebooks/cquangnguynl/nb2-pilot-dict-noise/syllable_table.json (tùy chọn)
  eval_report.json    : /kaggle/input/notebooks/cquangnguynl/nb3-baseline-train-eval/eval_report.json (tùy chọn)
Device: CUDA Tesla T4 | FP16=True


In [2]:
import re

SHARED_CELLS_VERSION = 'align-v1'

_WS_RE = re.compile(r'\s+')
_TOKEN_RE = re.compile(r'\w+|[^\w\s]+')
_HAS_WORD_RE = re.compile(r'\w')
_DIGIT_RE = re.compile(r'\d')

def nfc_normalize(s):
    return _WS_RE.sub(' ', unicodedata.normalize('NFC', s)).strip()

def canon_tokenize(s):
    """NFC + tách token: run chữ/số liền kề (\\w+) là 1 token; cụm dấu câu liền kề là 1 token riêng."""
    return _TOKEN_RE.findall(nfc_normalize(s))

def is_punct_token(tok):
    return not _HAS_WORD_RE.search(tok)

def is_word_token(tok):
    return not is_punct_token(tok) and not _DIGIT_RE.search(tok)

def levenshtein_opcodes(src, tgt):
    n, m = len(src), len(tgt)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        dp[i][0] = i
    for j in range(1, m + 1):
        dp[0][j] = j
    for i in range(1, n + 1):
        si = src[i - 1]
        prev, row = dp[i - 1], dp[i]
        for j in range(1, m + 1):
            best = prev[j - 1] + (0 if si == tgt[j - 1] else 1)
            if prev[j] + 1 < best:
                best = prev[j] + 1
            if row[j - 1] + 1 < best:
                best = row[j - 1] + 1
            row[j] = best
    ops = []
    def _push(tag, i1, i2, j1, j2):
        if ops and ops[-1][0] == tag and ops[-1][1] == i2 and ops[-1][3] == j2:
            ops[-1] = (tag, i1, ops[-1][2], j1, ops[-1][4])
        else:
            ops.append((tag, i1, i2, j1, j2))
    i, j = n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0 and dp[i][j] == dp[i - 1][j - 1] + (0 if src[i - 1] == tgt[j - 1] else 1):
            _push('equal' if src[i - 1] == tgt[j - 1] else 'replace', i - 1, i, j - 1, j)
            i, j = i - 1, j - 1
        elif i > 0 and dp[i][j] == dp[i - 1][j] + 1:
            _push('delete', i - 1, i, j, j)
            i -= 1
        else:
            _push('insert', i, i, j - 1, j)
            j -= 1
    ops.reverse()
    return ops

def _make_block(src, tgt, span):
    i1, i2, j1, j2 = span
    src_toks, tgt_toks = src[i1:i2], tgt[j1:j2]
    ns, nt = len(src_toks), len(tgt_toks)
    if ns == 0:
        btype = 'insert'
    elif nt == 0:
        btype = 'delete'
    elif ns == 1 and nt == 1:
        btype = 'substitute'
    elif ns == 1:
        btype = 'split'
    elif nt == 1:
        btype = 'merge'
    else:
        btype = 'multi'
    return {
        'type': btype,
        'position': i1,
        'src_span': [i1, i2],
        'tgt_span': [j1, j2],
        'src_tokens': src_toks,
        'tgt_tokens': tgt_toks,
        'punct_only': all(is_punct_token(t) for t in src_toks + tgt_toks),
    }

def extract_edit_blocks(src, tgt, opcodes):
    blocks, cur = [], None
    for tag, i1, i2, j1, j2 in opcodes:
        if tag == 'equal':
            if cur is not None:
                blocks.append(_make_block(src, tgt, cur))
                cur = None
        elif cur is None:
            cur = [i1, i2, j1, j2]
        else:
            cur[1], cur[3] = i2, j2
    if cur is not None:
        blocks.append(_make_block(src, tgt, cur))
    return blocks

def apply_edit_blocks(src, blocks):
    out, pos = [], 0
    for b in blocks:
        out += src[pos:b['src_span'][0]]
        out += b['tgt_tokens']
        pos = b['src_span'][1]
    out += src[pos:]
    return out

def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

print('SHARED_CELLS_VERSION:', SHARED_CELLS_VERSION)


SHARED_CELLS_VERSION: align-v1


In [3]:
def evaluate_predictions(records, predictions, syll_set=None):
    assert len(records) == len(predictions), f'Độ dài không khớp: {len(records)} vs {len(predictions)}'
    
    total_tp = 0
    total_fp = 0
    total_fn = 0
    correct_at_tp = 0
    
    total_clean_tokens = 0
    clean_sents_total = 0
    clean_sents_preserved = 0
    
    # Stratified stats: non-word vs real-word
    stratified = {
        'nonword': {'gold': 0, 'detected': 0, 'corrected': 0},
        'realword': {'gold': 0, 'detected': 0, 'corrected': 0}
    }
    
    sample_overcorrections = []
    
    for rec, pred in zip(records, predictions):
        src_toks = canon_tokenize(rec['text'])
        gold_toks = canon_tokenize(rec['corrected_text'])
        pred_toks = canon_tokenize(pred)
        
        # Gold edit blocks
        gold_ops = levenshtein_opcodes(src_toks, gold_toks)
        gold_blocks = extract_edit_blocks(src_toks, gold_toks, gold_ops)
        
        # Pred edit blocks
        pred_ops = levenshtein_opcodes(src_toks, pred_toks)
        pred_blocks = extract_edit_blocks(src_toks, pred_toks, pred_ops)
        
        gold_pos_map = {}
        for b in gold_blocks:
            if b['src_span'][0] < b['src_span'][1]:
                target_str = ' '.join(b['tgt_tokens'])
                for p in range(b['src_span'][0], b['src_span'][1]):
                    gold_pos_map[p] = (target_str, b['src_tokens'])
                    
        pred_pos_map = {}
        for b in pred_blocks:
            if b['src_span'][0] < b['src_span'][1]:
                pred_str = ' '.join(b['tgt_tokens'])
                for p in range(b['src_span'][0], b['src_span'][1]):
                    pred_pos_map[p] = (pred_str, b['src_tokens'])
                    
        gold_positions = set(gold_pos_map.keys())
        pred_positions = set(pred_pos_map.keys())
        
        tp_pos = gold_positions & pred_positions
        fp_pos = pred_positions - gold_positions
        fn_pos = gold_positions - pred_positions
        
        total_tp += len(tp_pos)
        total_fp += len(fp_pos)
        total_fn += len(fn_pos)
        
        # Đếm số token đúng trong câu nguồn (loại bỏ token dấu câu)
        word_token_positions = {i for i, t in enumerate(src_toks) if is_word_token(t)}
        clean_word_positions = word_token_positions - gold_positions
        total_clean_tokens += len(clean_word_positions)
        
        # Check clean sentence
        if len(gold_positions) == 0:
            clean_sents_total += 1
            if len(pred_positions) == 0:
                clean_sents_preserved += 1
                
        # Correction accuracy
        for p in tp_pos:
            target_str, _ = gold_pos_map[p]
            pred_str, _ = pred_pos_map[p]
            if pred_str.lower() == target_str.lower():
                correct_at_tp += 1
                
            # Stratified
            if syll_set is not None and p < len(src_toks):
                tok_lower = src_toks[p].lower()
                cat = 'nonword' if tok_lower not in syll_set else 'realword'
                stratified[cat]['detected'] += 1
                if pred_str.lower() == target_str.lower():
                    stratified[cat]['corrected'] += 1
                    
        for p in fn_pos:
            if syll_set is not None and p < len(src_toks):
                tok_lower = src_toks[p].lower()
                cat = 'nonword' if tok_lower not in syll_set else 'realword'
                stratified[cat]['gold'] += 1
        for p in tp_pos:
            if syll_set is not None and p < len(src_toks):
                tok_lower = src_toks[p].lower()
                cat = 'nonword' if tok_lower not in syll_set else 'realword'
                stratified[cat]['gold'] += 1
                
        # Lưu mẫu over-correction tiêu biểu
        if fp_pos and len(sample_overcorrections) < 20:
            for p in sorted(fp_pos):
                if p < len(src_toks):
                    sample_overcorrections.append({
                        'orig_token': src_toks[p],
                        'pred_token': pred_pos_map[p][0],
                        'context': ' '.join(src_toks[max(0, p-3):min(len(src_toks), p+4)]),
                        'full_src': rec['text'],
                        'full_pred': pred
                    })
                    if len(sample_overcorrections) >= 20:
                        break

    precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    corr_acc = correct_at_tp / total_tp if total_tp > 0 else 0.0
    overcorr_rate = total_fp / total_clean_tokens if total_clean_tokens > 0 else 0.0
    clean_retention = clean_sents_preserved / clean_sents_total if clean_sents_total > 0 else 1.0

    return {
        'detection': {
            'tp': total_tp,
            'fp': total_fp,
            'fn': total_fn,
            'precision': precision,
            'recall': recall,
            'f1': f1,
        },
        'correction': {
            'correct_at_tp': correct_at_tp,
            'accuracy': corr_acc,
        },
        'over_correction': {
            'fp_count': total_fp,
            'clean_tokens': total_clean_tokens,
            'rate': overcorr_rate,
            'clean_sents_total': clean_sents_total,
            'clean_sents_preserved': clean_sents_preserved,
            'clean_retention_rate': clean_retention,
        },
        'stratified': stratified,
        'sample_overcorrections': sample_overcorrections,
    }

# Sanity Test hàm đánh giá trên ví dụ giả định
_test_recs = [{'text': 'học sanh đi hoc', 'corrected_text': 'học sinh đi học'}]
_test_preds = ['học sinh đi hoc'] # Model sửa 'sanh' -> 'sinh', nhưng bỏ sót 'hoc'
_m_test = evaluate_predictions(_test_recs, _test_preds)
assert _m_test['detection']['tp'] == 1 and _m_test['detection']['fn'] == 1 and _m_test['detection']['fp'] == 0
assert _m_test['correction']['accuracy'] == 1.0
print('Sanity check hàm evaluate_predictions PASS 100%!')


Sanity check hàm evaluate_predictions PASS 100%!


In [4]:
val_records = load_jsonl(INPUT_FILES['vsec_val.jsonl'])
test_records = load_jsonl(INPUT_FILES['test_aligned.jsonl'])

val_texts = [r['text'] for r in val_records]
test_texts = [r['text'] for r in test_records]

# Tùy chọn 1: bảng âm tiết từ nb2 (cho phân tích stratified non-word vs real-word)
SYLL_SET = None
if 'syllable_table.json' in INPUT_FILES:
    syllable_table_dict = json.loads(Path(INPUT_FILES['syllable_table.json']).read_text(encoding='utf-8'))
    SYLL_SET = set(syllable_table_dict['entries'])
    print(f'Đã nạp syllable_table.json: {len(SYLL_SET)} âm tiết (bật phân tích stratified).')
else:
    print('Không có syllable_table.json → bỏ qua phân tích stratified.')

# Tùy chọn 2: eval_report.json từ nb3 (cho bảng so sánh 4-way)
nb3_report = None
if 'eval_report.json' in INPUT_FILES:
    _rep = json.loads(Path(INPUT_FILES['eval_report.json']).read_text(encoding='utf-8'))
    _res = _rep.get('results') or {}
    if all(k in _res for k in ['val_run1_pure', 'val_run2_aug', 'test_run1_pure', 'test_run2_aug']):
        nb3_report = _rep
        print('Đã nạp eval_report.json của nb3 (bật bảng so sánh 4-way).')
    else:
        print('eval_report.json không chứa kết quả Run 1/Run 2 của nb3 → bảng chỉ có Identity | Zero-shot.')
else:
    print('Không có eval_report.json → bảng so sánh chỉ có Identity | Zero-shot (attach output của nb3 để có đủ 4 cột).')

print(f'Số lượng câu nạp vào — Val: {len(val_records)} | Test: {len(test_records)}')


Đã nạp syllable_table.json: 7884 âm tiết (bật phân tích stratified).
Đã nạp eval_report.json của nb3 (bật bảng so sánh 4-way).
Số lượng câu nạp vào — Val: 927 | Test: 5983


In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def batch_generate(model, tokenizer, texts, batch_size=16, beam_size=EVAL_BEAM_SIZE):
    model.eval()
    device = next(model.parameters()).device
    preds = []
    
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        enc = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=MAX_SOURCE_LEN,
            return_tensors='pt'
        ).to(device)
        
        with torch.no_grad():
            outputs = model.generate(
                input_ids=enc['input_ids'],
                attention_mask=enc['attention_mask'],
                max_length=MAX_TARGET_LEN,
                num_beams=beam_size,
                early_stopping=True if beam_size > 1 else False
            )
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        preds.extend([nfc_normalize(d) for d in decoded])
        
        if (i // batch_size) % 50 == 0:
            print(f'  Generated {min(i + batch_size, len(texts))}/{len(texts)} sentences...')
    return preds


config.json:   0%|          | 0.00/897 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

dict.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/2.83M [00:00<?, ?B/s]

In [6]:
# Identity baseline: dự đoán = câu nguồn (không sửa gì) — sàn (floor) lý thuyết
preds_val_identity = list(val_texts)
preds_test_identity = list(test_texts)

eval_val_identity = evaluate_predictions(val_records, preds_val_identity, SYLL_SET)
eval_test_identity = evaluate_predictions(test_records, preds_test_identity, SYLL_SET)

# Assert floor: pred == src ⇒ không có TP/FP nào, mọi chỉ số chạm sàn
for _name, _m in [('VAL', eval_val_identity), ('TEST', eval_test_identity)]:
    assert _m['detection']['tp'] == 0 and _m['detection']['fp'] == 0, f'Identity floor violation (TP/FP) trên {_name}'
    assert _m['detection']['precision'] == 0.0 and _m['detection']['recall'] == 0.0 and _m['detection']['f1'] == 0.0, f'Identity floor violation (P/R/F1) trên {_name}'
    assert _m['correction']['accuracy'] == 0.0, f'Identity floor violation (Corr Acc) trên {_name}'
    assert _m['over_correction']['rate'] == 0.0, f'Identity floor violation (Over-corr) trên {_name}'
    assert _m['over_correction']['clean_retention_rate'] == 1.0, f'Identity floor violation (Clean Retention) trên {_name}'

print('Identity floor check PASS: P=R=F1=0%, Corr Acc=0%, Over-corr=0%, Clean Retention=100%')
print()
for _name, _m, _n in [('VAL', eval_val_identity, len(val_records)), ('TEST', eval_test_identity, len(test_records))]:
    _d, _oc = _m['detection'], _m['over_correction']
    print(f'{_name}: {_n} câu | Gold edits (TP+FN) = {_d["tp"] + _d["fn"]} | Clean sents = {_oc["clean_sents_total"]} (giữ nguyên {_oc["clean_sents_preserved"]}/{_oc["clean_sents_total"]})')
print('Lưu ý: VSEC có has_errors=100% ⇒ số câu sạch rất nhỏ, Clean Retention phải đọc kèm con số tuyệt đối.')


Identity floor check PASS: P=R=F1=0%, Corr Acc=0%, Over-corr=0%, Clean Retention=100%

VAL: 927 câu | Gold edits (TP+FN) = 1143 | Clean sents = 3 (giữ nguyên 3/3)
TEST: 5983 câu | Gold edits (TP+FN) = 24885 | Clean sents = 653 (giữ nguyên 653/653)
Lưu ý: VSEC có has_errors=100% ⇒ số câu sạch rất nhỏ, Clean Retention phải đọc kèm con số tuyệt đối.


In [7]:
print(f'=== NẠP BASE MODEL {MODEL_NAME} (ZERO-SHOT: KHÔNG LoRA, KHÔNG TRAIN) ===')
if FP16:
    print('Nạp model torch_dtype=float16...')
    model_zs = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16)
else:
    print('Cảnh báo: không có CUDA → nạp model float32 (chạy sẽ rất chậm).')
    model_zs = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_zs.to(device)
model_zs.eval()
print(f'Model sẵn sàng trên: {next(model_zs.parameters()).device}')

print('--- INFERENCE ZERO-SHOT TRÊN VSEC-VAL ---')
preds_val_zs = batch_generate(model_zs, tokenizer, val_texts, batch_size=BATCH_GEN)

eval_val_zeroshot = evaluate_predictions(val_records, preds_val_zs, SYLL_SET)

_vz = eval_val_zeroshot
print('=== TÓM TẮT ZERO-SHOT VAL ===')
print(f"Detection P/R/F1: {_vz['detection']['precision']:.2%} / {_vz['detection']['recall']:.2%} / {_vz['detection']['f1']:.2%}")
print(f"Correction Acc @TP: {_vz['correction']['accuracy']:.2%} ({_vz['correction']['correct_at_tp']}/{_vz['detection']['tp']} TP)")
print(f"Over-correction: {_vz['over_correction']['rate']:.2%} ({_vz['over_correction']['fp_count']} FP / {_vz['over_correction']['clean_tokens']} token sạch)")
print(f"Clean Retention: {_vz['over_correction']['clean_retention_rate']:.2%} (tuyệt đối: {_vz['over_correction']['clean_sents_preserved']}/{_vz['over_correction']['clean_sents_total']} câu sạch)")


`torch_dtype` is deprecated! Use `dtype` instead!


=== NẠP BASE MODEL vinai/bartpho-syllable (ZERO-SHOT: KHÔNG LoRA, KHÔNG TRAIN) ===
Nạp model torch_dtype=float16...


pytorch_model.bin:   0%|          | 0.00/1.58G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/1.58G [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Model sẵn sàng trên: cuda:0
--- INFERENCE ZERO-SHOT TRÊN VSEC-VAL ---
  Generated 16/927 sentences...
  Generated 816/927 sentences...
=== TÓM TẮT ZERO-SHOT VAL ===
Detection P/R/F1: 29.49% / 5.60% / 9.41%
Correction Acc @TP: 28.12% (18/64 TP)
Over-correction: 0.57% (153 FP / 26615 token sạch)
Clean Retention: 0.00% (tuyệt đối: 0/3 câu sạch)


In [8]:
print('--- INFERENCE ZERO-SHOT TRÊN TEST ALIGNED (~6.000 câu, ~15–25 phút) ---')
preds_test_zs = batch_generate(model_zs, tokenizer, test_texts, batch_size=BATCH_GEN)

eval_test_zeroshot = evaluate_predictions(test_records, preds_test_zs, SYLL_SET)

_tz = eval_test_zeroshot
print('=== TÓM TẮT ZERO-SHOT TEST ===')
print(f"Detection P/R/F1: {_tz['detection']['precision']:.2%} / {_tz['detection']['recall']:.2%} / {_tz['detection']['f1']:.2%}")
print(f"Correction Acc @TP: {_tz['correction']['accuracy']:.2%} ({_tz['correction']['correct_at_tp']}/{_tz['detection']['tp']} TP)")
print(f"Over-correction: {_tz['over_correction']['rate']:.2%} ({_tz['over_correction']['fp_count']} FP / {_tz['over_correction']['clean_tokens']} token sạch)")
print(f"Clean Retention: {_tz['over_correction']['clean_retention_rate']:.2%} (tuyệt đối: {_tz['over_correction']['clean_sents_preserved']}/{_tz['over_correction']['clean_sents_total']} câu sạch)")


--- INFERENCE ZERO-SHOT TRÊN TEST ALIGNED (~6.000 câu, ~15–25 phút) ---
  Generated 16/5983 sentences...
  Generated 816/5983 sentences...
  Generated 1616/5983 sentences...
  Generated 2416/5983 sentences...
  Generated 3216/5983 sentences...
  Generated 4016/5983 sentences...
  Generated 4816/5983 sentences...
  Generated 5616/5983 sentences...
=== TÓM TẮT ZERO-SHOT TEST ===
Detection P/R/F1: 50.85% / 9.88% / 16.55%
Correction Acc @TP: 13.91% (342/2459 TP)
Over-correction: 1.56% (2377 FP / 152815 token sạch)
Clean Retention: 83.77% (tuyệt đối: 547/653 câu sạch)


In [9]:
_nb3_res = nb3_report['results'] if nb3_report else {}
run1_val = _nb3_res.get('val_run1_pure')
run2_val = _nb3_res.get('val_run2_aug')
run1_test = _nb3_res.get('test_run1_pure')
run2_test = _nb3_res.get('test_run2_aug')

def print_compare_table(split_label, m_identity, m_zeroshot, m_run1=None, m_run2=None):
    cols = [('Identity', m_identity), ('Zero-shot', m_zeroshot)]
    if m_run1 is not None:
        cols.append(('Run 1', m_run1))
    if m_run2 is not None:
        cols.append(('Run 2', m_run2))

    def _cell(m, key, sub):
        return f'{m[key][sub]:.2%}' if m is not None else 'N/A'

    header = f'{"CHỈ SỐ ĐÁNH GIÁ":<30s} | ' + ' | '.join(f'{split_label}: {name}' for name, _ in cols)
    print('=' * len(header))
    print(header)
    print('-' * len(header))
    rows = [
        ('Detection Precision', 'detection', 'precision'),
        ('Detection Recall', 'detection', 'recall'),
        ('Detection F1-Score', 'detection', 'f1'),
        ('Correction Accuracy (at TP)', 'correction', 'accuracy'),
        ('Over-correction Rate', 'over_correction', 'rate'),
        ('Clean Sent Retention Rate', 'over_correction', 'clean_retention_rate'),
    ]
    for name, key, sub in rows:
        cells = ' | '.join(f'{_cell(m, key, sub):<16s}' for _, m in cols)
        print(f'{name:<30s} | {cells}')
    print('=' * len(header))

print('=== BẢNG SO SÁNH — VSEC-VAL ===')
print_compare_table('VAL', eval_val_identity, eval_val_zeroshot, run1_val, run2_val)
print()
print('=== BẢNG SO SÁNH — TEST ALIGNED ===')
print_compare_table('TEST', eval_test_identity, eval_test_zeroshot, run1_test, run2_test)
if nb3_report is None:
    print('(Ghi chú: thiếu eval_report.json của nb3 → chỉ có cột Identity | Zero-shot. Attach output của nb3 làm Input để có đủ 4 cột.)')

print()
print('=== 5 MẪU OVER-CORRECTION (FP) CỦA ZERO-SHOT TRÊN VAL ===')
print('(Đối chiếu: FP của Run 1 chủ yếu là deletion — token đúng bị xóa thay vì thay thế)')
for s in eval_val_zeroshot['sample_overcorrections'][:5]:
    print(f"  Từ đúng: '{s['orig_token']}' -> Bị sửa thành: '{s['pred_token']}'")
    print(f"  Ngữ cảnh: ...{s['context']}...")
    print()

print('=== PHÂN TÍCH STRATIFIED (VSEC-VAL) ===')
if SYLL_SET is not None:
    def _cat_recall(m, cat):
        s = m['stratified'][cat]
        return s['detected'] / s['gold'] if s['gold'] else 0.0
    for cat in ['nonword', 'realword']:
        parts = [f'Identity: {_cat_recall(eval_val_identity, cat):.2%}',
                 f'Zero-shot: {_cat_recall(eval_val_zeroshot, cat):.2%}']
        if run1_val is not None:
            parts.append(f'Run 1: {_cat_recall(run1_val, cat):.2%}')
        if run2_val is not None:
            parts.append(f'Run 2: {_cat_recall(run2_val, cat):.2%}')
        print(f'Loại lỗi [{cat.upper():8s}] — Gold: {eval_val_zeroshot["stratified"][cat]["gold"]} lỗi | ' + ' | '.join(parts))
else:
    print('Bỏ qua (không có syllable_table.json).')


=== BẢNG SO SÁNH — VSEC-VAL ===
CHỈ SỐ ĐÁNH GIÁ                | VAL: Identity | VAL: Zero-shot | VAL: Run 1 | VAL: Run 2
-----------------------------------------------------------------------------------------
Detection Precision            | 0.00%            | 29.49%           | 80.51%           | 79.00%          
Detection Recall               | 0.00%            | 5.60%            | 72.27%           | 70.78%          
Detection F1-Score             | 0.00%            | 9.41%            | 76.16%           | 74.67%          
Correction Accuracy (at TP)    | 0.00%            | 28.12%           | 84.75%           | 86.53%          
Over-correction Rate           | 0.00%            | 0.57%            | 0.75%            | 0.81%           
Clean Sent Retention Rate      | 100.00%          | 0.00%            | 33.33%           | 33.33%          

=== BẢNG SO SÁNH — TEST ALIGNED ===
CHỈ SỐ ĐÁNH GIÁ                | TEST: Identity | TEST: Zero-shot | TEST: Run 1 | TEST: Run 2
---------------

In [10]:
def save_predictions_jsonl(records, preds, out_path):
    with open(out_path, 'w', encoding='utf-8') as f:
        for r, p in zip(records, preds):
            item = {
                'row_id': r.get('row_id'),
                'text': r['text'],
                'corrected_text': r['corrected_text'],
                'prediction': p
            }
            f.write(json.dumps(item, ensure_ascii=False) + '\n')

save_predictions_jsonl(val_records, preds_val_zs, OUTPUT_DIR / 'predictions_val_zeroshot.jsonl')
save_predictions_jsonl(test_records, preds_test_zs, OUTPUT_DIR / 'predictions_test_zeroshot.jsonl')

zeroshot_eval_report = {
    'created': RUN_STAMP,
    'notebook': 'nb3b_zeroshot_baseline',
    'shared_cells_version': SHARED_CELLS_VERSION,
    'config': {
        'seed': SEED,
        'model_name': MODEL_NAME,
        'eval_beam_size': EVAL_BEAM_SIZE,
        'max_source_len': MAX_SOURCE_LEN,
        'max_target_len': MAX_TARGET_LEN,
        'batch_gen': BATCH_GEN,
        'fp16': FP16,
        'lora': False,
        'training': False
    },
    'results': {
        'val_identity': eval_val_identity,
        'test_identity': eval_test_identity,
        'val_zeroshot': eval_val_zeroshot,
        'test_zeroshot': eval_test_zeroshot
    }
}

if nb3_report is not None:
    # Khối tham chiếu: chép nguyên 4 kết quả Run 1/Run 2 từ eval_report.json của nb3
    # để file này tự chứa đủ ngữ cảnh so sánh (một nguồn sự thật — không tính lại).
    zeroshot_eval_report['nb3_reference'] = {
        'source': 'eval_report.json (nb3_baseline_train_eval)',
        'created': nb3_report.get('created'),
        'results': {
            'val_run1_pure': nb3_report['results']['val_run1_pure'],
            'val_run2_aug': nb3_report['results']['val_run2_aug'],
            'test_run1_pure': nb3_report['results']['test_run1_pure'],
            'test_run2_aug': nb3_report['results']['test_run2_aug']
        }
    }

with open(OUTPUT_DIR / 'zeroshot_eval_report.json', 'w', encoding='utf-8') as f:
    json.dump(zeroshot_eval_report, f, ensure_ascii=False, indent=2)

print('=== HOÀN TẤT XUẤT ĐẦU RA ===')
print('Các file đã ghi vào', OUTPUT_DIR)
for fn in ['predictions_val_zeroshot.jsonl', 'predictions_test_zeroshot.jsonl', 'zeroshot_eval_report.json']:
    print(' -', fn)


=== HOÀN TẤT XUẤT ĐẦU RA ===
Các file đã ghi vào /kaggle/working
 - predictions_val_zeroshot.jsonl
 - predictions_test_zeroshot.jsonl
 - zeroshot_eval_report.json
